# Goodreads Reviews — Interactive EDA Notebook

Notebook này rebuild lại EDA theo strategy mới:

- DuckDB đọc trực tiếp Parquet từ Hugging Face hoặc local path.
- Pandas chỉ nhận aggregate/sample nhỏ, không load toàn bộ raw dataset.
- Biểu đồ chính dùng Plotly interactive.
- Tự động lưu chart thành `eda_outputs/*.html` để nhúng web demo.
- Tự động lưu bảng aggregate thành CSV/JSON.
- Có phần optional cho dataset đã concat/preprocess có `primary_genre`, `split`, `is_bridge_user`.

## Pipeline EDA chính

1. Overview + sparsity
2. Missingness/data quality
3. Rating distribution
4. Reviews/users theo năm và theo tháng
5. User growth
6. Reviews per user
7. Reviews per book
8. Book long-tail
9. Rating bias theo review volume
10. Review length
11. Rating drift theo thời gian
12. Genre/bridge/split diagnostics nếu có cột tương ứng

In [ ]:
# Cell 0 — Install dependencies
# Kaggle: bật Internet nếu đọc hf:// trực tiếp từ Hugging Face.
!pip install duckdb plotly pandas numpy pyarrow -q

In [ ]:
# Cell 1 — Imports, config, helpers

import os
import json
import time
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
# Raw reviews:
#   DATASET_REPO = "vngclinh/goodreads-reviews"
#   PARQUET_GLOB = "data/*.parquet"
#
# Concat dataset có metadata + primary_genre:
#   DATASET_REPO = "vngclinh/goodreads-concats"
#   PARQUET_GLOB = "data/**/*.parquet"
#
# Preprocessed dataset:
#   DATASET_REPO = "vngclinh/goodreads-preprocessed"
#   PARQUET_GLOB = "data/**/*.parquet"
#
# Nếu chạy local, thay DATASET_URI bằng path local, ví dụ:
#   DATASET_URI = "D:/goodreads/preprocessed/*.parquet"

DATASET_REPO = "vngclinh/goodreads-preprocessed"
PARQUET_GLOB = "data/**/*.parquet"
DATASET_URI  = f"hf://datasets/{DATASET_REPO}/{PARQUET_GLOB}"

OUTPUT_DIR = Path("eda_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_LENGTH_SAMPLE_N = 300_000
RUN_GENRE_ANALYSIS = True
RUN_GENRE_NETWORK  = True

ACCENT = "#378ADD"
MUTED  = "#888780"
WARM   = "#D85A30"
GREEN  = "#1D9E75"

saved_artifacts = []

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

def run(sql, label="query"):
    t0 = time.time()
    df = con.execute(sql).df()
    elapsed = time.time() - t0
    log(f"{label}: {len(df):,} rows · {elapsed:.2f}s")
    return df

def save_fig(fig, name, title=None, height=None, width=None):
    if title:
        fig.update_layout(title=title)
    if height:
        fig.update_layout(height=height)
    if width:
        fig.update_layout(width=width)
    fig.update_layout(
        template="plotly_white",
        margin=dict(l=50, r=30, t=70, b=50),
        hovermode="closest"
    )
    path = OUTPUT_DIR / f"{name}.html"
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    saved_artifacts.append((name, f"{name}.html"))
    fig.show()
    print(f"Saved → {path}")
    return path

def save_table(df, name):
    csv_path = OUTPUT_DIR / f"{name}.csv"
    json_path = OUTPUT_DIR / f"{name}.json"
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", force_ascii=False, indent=2)
    saved_artifacts.append((f"{name} data", f"{name}.csv"))
    print(f"Saved table → {csv_path}")
    return csv_path

def compact_int(x):
    if pd.isna(x):
        return "NA"
    x = float(x)
    if x >= 1e9:
        return f"{x/1e9:.2f}B"
    if x >= 1e6:
        return f"{x/1e6:.2f}M"
    if x >= 1e3:
        return f"{x/1e3:.2f}K"
    return f"{x:.0f}"

In [ ]:
# Cell 2 — Connect DuckDB and inspect schema

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

BASE_SCAN = f"read_parquet('{DATASET_URI}', union_by_name=true, hive_partitioning=true)"

schema = run(f"DESCRIBE SELECT * FROM {BASE_SCAN} LIMIT 1", "schema")
display(schema)

cols = set(schema["column_name"].tolist())
col_types = dict(zip(schema["column_name"], schema["column_type"]))

required = {"user_id", "book_id", "rating", "review_text", "date_added"}
missing_required = required - cols
if missing_required:
    raise ValueError(f"Dataset thiếu các cột bắt buộc: {missing_required}")

print("Dataset URI:", DATASET_URI)
print("Columns:", sorted(cols))

In [ ]:
# Cell 3 — Robust date expression

# Raw dataset thường lưu date_added là string kiểu Goodreads:
# Thu Jan 01 00:00:00 -0800 2015
# Preprocessed dataset có thể là TIMESTAMP.

date_type = col_types.get("date_added", "").upper()

if any(t in date_type for t in ["TIMESTAMP", "DATE"]):
    DATE_TS = "CAST(date_added AS TIMESTAMP)"
else:
    DATE_TS = """
    COALESCE(
        try_strptime(CAST(date_added AS VARCHAR), '%a %b %d %H:%M:%S %z %Y'),
        try_strptime(CAST(date_added AS VARCHAR), '%Y-%m-%d %H:%M:%S%z'),
        try_strptime(CAST(date_added AS VARCHAR), '%Y-%m-%d %H:%M:%S'),
        try_cast(date_added AS TIMESTAMP)
    )
    """

YEAR_EXPR  = f"YEAR({DATE_TS})"
MONTH_EXPR = f"DATE_TRUNC('month', {DATE_TS})"
DATE_EXPR  = f"CAST({DATE_TS} AS DATE)"

print("date_added type:", date_type)
print("DATE_TS expression is ready.")

In [ ]:
# Cell 4 — Dataset overview + recommender core metrics

overview = run(f"""
WITH base AS (
    SELECT
        user_id,
        book_id,
        CAST(rating AS DOUBLE) AS rating,
        review_text,
        {DATE_TS} AS date_ts
    FROM {BASE_SCAN}
),
agg AS (
    SELECT
        COUNT(*) AS total_reviews,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT book_id) AS unique_books,
        MIN(CAST(date_ts AS DATE)) AS earliest_date,
        MAX(CAST(date_ts AS DATE)) AS latest_date,
        AVG(CASE WHEN rating BETWEEN 1 AND 5 THEN rating END) AS avg_rating,
        SUM(CASE WHEN review_text IS NOT NULL AND LENGTH(TRIM(review_text)) > 0 THEN 1 ELSE 0 END) AS reviews_with_text
    FROM base
)
SELECT
    *,
    total_reviews * 1.0 / NULLIF(unique_users, 0) AS avg_reviews_per_user,
    total_reviews * 1.0 / NULLIF(unique_books, 0) AS avg_reviews_per_book,
    total_reviews * 1.0 / NULLIF(unique_users * unique_books, 0) AS matrix_density,
    1 - total_reviews * 1.0 / NULLIF(unique_users * unique_books, 0) AS matrix_sparsity,
    reviews_with_text * 100.0 / NULLIF(total_reviews, 0) AS pct_reviews_with_text
FROM agg
""", "overview")

row = overview.iloc[0].to_dict()

summary_metrics = {
    "dataset_repo": DATASET_REPO,
    "dataset_uri": DATASET_URI,
    "total_reviews": int(row["total_reviews"]),
    "unique_users": int(row["unique_users"]),
    "unique_books": int(row["unique_books"]),
    "earliest_date": str(row["earliest_date"]),
    "latest_date": str(row["latest_date"]),
    "avg_rating": None if pd.isna(row["avg_rating"]) else float(row["avg_rating"]),
    "avg_reviews_per_user": float(row["avg_reviews_per_user"]),
    "avg_reviews_per_book": float(row["avg_reviews_per_book"]),
    "matrix_density": float(row["matrix_density"]),
    "matrix_sparsity": float(row["matrix_sparsity"]),
    "pct_reviews_with_text": float(row["pct_reviews_with_text"]),
}

with open(OUTPUT_DIR / "summary_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, ensure_ascii=False, indent=2)

metrics = [
    ("Total reviews", compact_int(row["total_reviews"])),
    ("Unique users", compact_int(row["unique_users"])),
    ("Unique books", compact_int(row["unique_books"])),
    ("Avg rating", f"{row['avg_rating']:.2f}/5"),
    ("Sparsity", f"{row['matrix_sparsity']*100:.5f}%"),
    ("Reviews w/ text", f"{row['pct_reviews_with_text']:.1f}%"),
    ("Date range", f"{row['earliest_date']}<br>→ {row['latest_date']}"),
    ("Avg reviews/user", f"{row['avg_reviews_per_user']:.2f}"),
]

fig = go.Figure()
n = len(metrics)
for i, (label, val) in enumerate(metrics):
    x = (i + 0.5) / n
    fig.add_annotation(x=x, y=0.62, text=f"<b>{val}</b>", showarrow=False,
                       font=dict(size=21, color="#1f3556"), xanchor="center", align="center")
    fig.add_annotation(x=x, y=0.26, text=f"<span style='color:#777'>{label}</span>", showarrow=False,
                       font=dict(size=12), xanchor="center", align="center")

fig.update_layout(
    title="Dataset overview",
    height=250,
    width=1350,
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[0, 1]),
    plot_bgcolor="white"
)
save_fig(fig, "overview")
display(overview)

In [ ]:
# Cell 5 — Missingness / data quality by column

candidate_cols = [
    "user_id", "book_id", "review_id", "rating", "review_text", "date_added",
    "n_votes", "n_comments", "title", "description", "author_id",
    "publication_year", "num_pages", "primary_genre", "split"
]
check_cols = [c for c in candidate_cols if c in cols]

parts = []
for c in check_cols:
    parts.append(f"""
    SELECT
        '{c}' AS column_name,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS null_rows,
        SUM(CASE WHEN CAST({c} AS VARCHAR) = '' THEN 1 ELSE 0 END) AS empty_string_rows
    FROM {BASE_SCAN}
    """)

missingness = run(" UNION ALL ".join(parts), "missingness")
missingness["null_pct"] = missingness["null_rows"] / missingness["total_rows"] * 100
missingness["empty_string_pct"] = missingness["empty_string_rows"] / missingness["total_rows"] * 100

fig = px.bar(
    missingness.sort_values("null_pct", ascending=False),
    x="column_name", y="null_pct",
    hover_data=["null_rows", "empty_string_rows", "empty_string_pct"],
    labels={"column_name": "Column", "null_pct": "Null rows (%)"},
    title="Missingness by column"
)
fig.update_traces(marker_color=ACCENT)
save_fig(fig, "missingness")
save_table(missingness, "missingness")
display(missingness)

In [ ]:
# Cell 6 — Rating distribution

rating_dist = run(f"""
SELECT
    CAST(rating AS INTEGER) AS rating,
    COUNT(*) AS n
FROM {BASE_SCAN}
WHERE CAST(rating AS INTEGER) BETWEEN 1 AND 5
GROUP BY 1
ORDER BY 1
""", "rating distribution")

rating_dist["pct"] = rating_dist["n"] / rating_dist["n"].sum() * 100
mean_rating = (rating_dist["rating"] * rating_dist["n"]).sum() / rating_dist["n"].sum()

fig = px.bar(
    rating_dist,
    x="rating", y="n",
    text=rating_dist["pct"].map(lambda x: f"{x:.1f}%"),
    color="rating",
    color_continuous_scale="Blues",
    labels={"rating": "Rating", "n": "Number of reviews"},
    title=f"Rating distribution · mean={mean_rating:.2f}"
)
fig.update_traces(textposition="outside")
fig.update_layout(coloraxis_showscale=False)
save_fig(fig, "rating_distribution", height=480)
save_table(rating_dist, "rating_distribution")
display(rating_dist)

In [ ]:
# Cell 7 — Reviews and active users by year

yearly = run(f"""
WITH base AS (
    SELECT
        user_id,
        CAST(rating AS DOUBLE) AS rating,
        {DATE_TS} AS date_ts,
        review_text
    FROM {BASE_SCAN}
),
valid AS (
    SELECT
        YEAR(date_ts) AS yr,
        user_id,
        rating,
        CASE WHEN review_text IS NOT NULL AND LENGTH(TRIM(review_text)) > 0 THEN 1 ELSE 0 END AS has_text
    FROM base
    WHERE date_ts IS NOT NULL
)
SELECT
    yr,
    COUNT(*) AS reviews,
    COUNT(DISTINCT user_id) AS active_users,
    AVG(CASE WHEN rating BETWEEN 1 AND 5 THEN rating END) AS avg_rating,
    SUM(has_text) * 100.0 / COUNT(*) AS pct_has_text
FROM valid
GROUP BY 1
ORDER BY 1
""", "yearly activity")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=yearly["yr"], y=yearly["reviews"], name="Reviews", marker_color=ACCENT, opacity=0.75), secondary_y=False)
fig.add_trace(go.Scatter(x=yearly["yr"], y=yearly["active_users"], name="Active users", mode="lines+markers", line=dict(color=WARM, width=3)), secondary_y=True)
fig.update_yaxes(title_text="Reviews", secondary_y=False)
fig.update_yaxes(title_text="Active users", secondary_y=True)
fig.update_xaxes(title_text="Year")
save_fig(fig, "reviews_and_users_by_year", title="Reviews and active users by year", height=500)
save_table(yearly, "yearly_activity")
display(yearly)

In [ ]:
# Cell 8 — User growth: new users by first review year

user_first_year = run(f"""
WITH u AS (
    SELECT
        user_id,
        MIN(YEAR({DATE_TS})) AS first_year
    FROM {BASE_SCAN}
    WHERE {DATE_TS} IS NOT NULL
    GROUP BY user_id
)
SELECT
    first_year AS yr,
    COUNT(*) AS new_users
FROM u
WHERE first_year IS NOT NULL
GROUP BY 1
ORDER BY 1
""", "new users")

user_first_year["cumulative_users"] = user_first_year["new_users"].cumsum()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=user_first_year["yr"], y=user_first_year["new_users"], name="New users", marker_color=ACCENT), secondary_y=False)
fig.add_trace(go.Scatter(x=user_first_year["yr"], y=user_first_year["cumulative_users"], name="Cumulative users", line=dict(color=GREEN, width=3)), secondary_y=True)
fig.update_xaxes(title_text="First active year")
fig.update_yaxes(title_text="New users", secondary_y=False)
fig.update_yaxes(title_text="Cumulative users", secondary_y=True)
save_fig(fig, "user_growth", title="User growth by first review year", height=500)
save_table(user_first_year, "user_growth")
display(user_first_year)

In [ ]:
# Cell 9 — Monthly reviews + avg rating

monthly = run(f"""
SELECT
    DATE_TRUNC('month', {DATE_TS}) AS month,
    COUNT(*) AS reviews,
    AVG(CASE WHEN CAST(rating AS INTEGER) BETWEEN 1 AND 5 THEN CAST(rating AS DOUBLE) END) AS avg_rating,
    COUNT(DISTINCT user_id) AS active_users
FROM {BASE_SCAN}
WHERE {DATE_TS} IS NOT NULL
GROUP BY 1
ORDER BY 1
""", "monthly activity")

monthly["reviews_ma3"] = monthly["reviews"].rolling(3, min_periods=1).mean()
monthly["rating_ma3"] = monthly["avg_rating"].rolling(3, min_periods=1).mean()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=monthly["month"], y=monthly["reviews"], name="Reviews / month", marker_color=ACCENT, opacity=0.55), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["reviews_ma3"], name="Reviews 3-month MA", line=dict(color=ACCENT, width=3)), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["rating_ma3"], name="Avg rating 3-month MA", line=dict(color=WARM, width=2)), secondary_y=True)
fig.update_yaxes(title_text="Reviews", secondary_y=False)
fig.update_yaxes(title_text="Avg rating", secondary_y=True)
fig.update_xaxes(title_text="Month")
save_fig(fig, "reviews_by_month", title="Monthly review volume and rating drift", height=500)
save_table(monthly, "monthly_activity")
display(monthly.head())

In [ ]:
# Cell 10 — Reviews per user distribution + CCDF

user_dist = run(f"""
WITH u AS (
    SELECT user_id, COUNT(*) AS n_reviews
    FROM {BASE_SCAN}
    GROUP BY user_id
),
d AS (
    SELECT n_reviews, COUNT(*) AS n_users
    FROM u
    GROUP BY n_reviews
)
SELECT *
FROM d
ORDER BY n_reviews
""", "reviews per user distribution")

user_dist["cum_users_from_right"] = user_dist["n_users"][::-1].cumsum()[::-1]
user_dist["ccdf"] = user_dist["cum_users_from_right"] / user_dist["n_users"].sum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Distribution", "CCDF: P(user reviews ≥ x)"])
fig.add_trace(go.Bar(x=user_dist["n_reviews"], y=user_dist["n_users"], marker_color=ACCENT), row=1, col=1)
fig.add_trace(go.Scatter(x=user_dist["n_reviews"], y=user_dist["ccdf"], mode="lines", line=dict(color=WARM, width=3)), row=1, col=2)
fig.update_xaxes(type="log", title_text="Reviews per user", row=1, col=1)
fig.update_yaxes(type="log", title_text="Users", row=1, col=1)
fig.update_xaxes(type="log", title_text="Reviews per user", row=1, col=2)
fig.update_yaxes(type="log", title_text="CCDF", row=1, col=2)
save_fig(fig, "reviews_per_user", title="Reviews per user: long-tail behavior", height=520, width=1250)
save_table(user_dist, "reviews_per_user_distribution")

user_stats = run(f"""
WITH u AS (
    SELECT user_id, COUNT(*) AS n_reviews
    FROM {BASE_SCAN}
    GROUP BY user_id
)
SELECT
    COUNT(*) AS users,
    AVG(n_reviews) AS mean_reviews,
    MEDIAN(n_reviews) AS median_reviews,
    QUANTILE_CONT(n_reviews, 0.90) AS p90_reviews,
    QUANTILE_CONT(n_reviews, 0.95) AS p95_reviews,
    QUANTILE_CONT(n_reviews, 0.99) AS p99_reviews,
    MAX(n_reviews) AS max_reviews,
    SUM(CASE WHEN n_reviews = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_one_review_users
FROM u
""", "user stats")
display(user_stats)
save_table(user_stats, "user_stats")

In [ ]:
# Cell 11 — User activity tiers

user_tiers = run(f"""
WITH u AS (
    SELECT
        user_id,
        COUNT(*) AS n_reviews,
        AVG(CASE WHEN CAST(rating AS INTEGER) BETWEEN 1 AND 5 THEN CAST(rating AS DOUBLE) END) AS avg_rating
    FROM {BASE_SCAN}
    GROUP BY user_id
),
tiered AS (
    SELECT
        *,
        CASE
            WHEN n_reviews = 1 THEN '1'
            WHEN n_reviews BETWEEN 2 AND 5 THEN '2–5'
            WHEN n_reviews BETWEEN 6 AND 20 THEN '6–20'
            WHEN n_reviews BETWEEN 21 AND 100 THEN '21–100'
            ELSE '100+'
        END AS activity_tier
    FROM u
)
SELECT
    activity_tier,
    COUNT(*) AS users,
    AVG(n_reviews) AS avg_reviews_per_user,
    AVG(avg_rating) AS avg_user_rating
FROM tiered
GROUP BY activity_tier
ORDER BY
    CASE activity_tier
        WHEN '1' THEN 1
        WHEN '2–5' THEN 2
        WHEN '6–20' THEN 3
        WHEN '21–100' THEN 4
        ELSE 5
    END
""", "user activity tiers")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=user_tiers["activity_tier"], y=user_tiers["users"], name="Users", marker_color=ACCENT, text=user_tiers["users"].map(compact_int)), secondary_y=False)
fig.add_trace(go.Scatter(x=user_tiers["activity_tier"], y=user_tiers["avg_user_rating"], name="Avg user rating", mode="lines+markers", line=dict(color=WARM, width=3)), secondary_y=True)
fig.update_yaxes(title_text="Users", type="log", secondary_y=False)
fig.update_yaxes(title_text="Avg user rating", secondary_y=True)
fig.update_xaxes(title_text="Activity tier: reviews written")
save_fig(fig, "user_activity_tiers", title="User activity tiers and rating tendency", height=500)
save_table(user_tiers, "user_activity_tiers")
display(user_tiers)

In [ ]:
# Cell 12 — Reviews per book distribution + CCDF

book_dist = run(f"""
WITH b AS (
    SELECT book_id, COUNT(*) AS n_reviews
    FROM {BASE_SCAN}
    GROUP BY book_id
),
d AS (
    SELECT n_reviews, COUNT(*) AS n_books
    FROM b
    GROUP BY n_reviews
)
SELECT *
FROM d
ORDER BY n_reviews
""", "reviews per book distribution")

book_dist["cum_books_from_right"] = book_dist["n_books"][::-1].cumsum()[::-1]
book_dist["ccdf"] = book_dist["cum_books_from_right"] / book_dist["n_books"].sum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Distribution", "CCDF: P(book reviews ≥ x)"])
fig.add_trace(go.Bar(x=book_dist["n_reviews"], y=book_dist["n_books"], marker_color=ACCENT), row=1, col=1)
fig.add_trace(go.Scatter(x=book_dist["n_reviews"], y=book_dist["ccdf"], mode="lines", line=dict(color=WARM, width=3)), row=1, col=2)
fig.update_xaxes(type="log", title_text="Reviews per book", row=1, col=1)
fig.update_yaxes(type="log", title_text="Books", row=1, col=1)
fig.update_xaxes(type="log", title_text="Reviews per book", row=1, col=2)
fig.update_yaxes(type="log", title_text="CCDF", row=1, col=2)
save_fig(fig, "reviews_per_book", title="Reviews per book: popularity long-tail", height=520, width=1250)
save_table(book_dist, "reviews_per_book_distribution")

book_stats = run(f"""
WITH b AS (
    SELECT book_id, COUNT(*) AS n_reviews
    FROM {BASE_SCAN}
    GROUP BY book_id
)
SELECT
    COUNT(*) AS books,
    AVG(n_reviews) AS mean_reviews,
    MEDIAN(n_reviews) AS median_reviews,
    QUANTILE_CONT(n_reviews, 0.90) AS p90_reviews,
    QUANTILE_CONT(n_reviews, 0.95) AS p95_reviews,
    QUANTILE_CONT(n_reviews, 0.99) AS p99_reviews,
    MAX(n_reviews) AS max_reviews,
    SUM(CASE WHEN n_reviews = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_one_review_books
FROM b
""", "book stats")
display(book_stats)
save_table(book_stats, "book_stats")

In [ ]:
# Cell 13 — Top reviewed books

title_expr = "ANY_VALUE(title) AS title," if "title" in cols else "CAST(NULL AS VARCHAR) AS title,"
author_expr = "ANY_VALUE(author_id) AS author_id," if "author_id" in cols else "CAST(NULL AS VARCHAR) AS author_id,"
genre_expr = "ANY_VALUE(primary_genre) AS primary_genre," if "primary_genre" in cols else "CAST(NULL AS VARCHAR) AS primary_genre,"

top_books = run(f"""
SELECT
    book_id,
    {title_expr}
    {author_expr}
    {genre_expr}
    COUNT(*) AS n_reviews,
    AVG(CASE WHEN CAST(rating AS INTEGER) BETWEEN 1 AND 5 THEN CAST(rating AS DOUBLE) END) AS avg_rating
FROM {BASE_SCAN}
GROUP BY book_id
ORDER BY n_reviews DESC
LIMIT 100
""", "top reviewed books")

label_col = "title" if "title" in cols else "book_id"
plot_top = top_books.head(30).copy()
plot_top["label"] = plot_top[label_col].fillna(plot_top["book_id"]).astype(str).str.slice(0, 60)

fig = px.bar(
    plot_top.sort_values("n_reviews"),
    x="n_reviews", y="label",
    orientation="h",
    color="avg_rating",
    color_continuous_scale="Blues",
    hover_data=["book_id", "primary_genre", "avg_rating"],
    labels={"n_reviews": "Reviews", "label": "Book", "avg_rating": "Avg rating"},
    title="Top reviewed books"
)
save_fig(fig, "top_reviewed_books", height=750)
save_table(top_books, "top_reviewed_books")
display(top_books.head(20))

In [ ]:
# Cell 14 — Book long-tail: cumulative review share by book rank

book_rank = run(f"""
WITH b AS (
    SELECT
        book_id,
        COUNT(*) AS n_reviews
    FROM {BASE_SCAN}
    GROUP BY book_id
),
ranked AS (
    SELECT
        book_id,
        n_reviews,
        ROW_NUMBER() OVER (ORDER BY n_reviews DESC) AS book_rank,
        SUM(n_reviews) OVER () AS total_reviews,
        SUM(n_reviews) OVER (ORDER BY n_reviews DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_reviews,
        COUNT(*) OVER () AS total_books
    FROM b
)
SELECT
    book_rank,
    n_reviews,
    book_rank * 100.0 / total_books AS pct_books,
    cumulative_reviews * 100.0 / total_reviews AS pct_reviews
FROM ranked
ORDER BY book_rank
""", "book long-tail")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=book_rank["pct_books"],
    y=book_rank["pct_reviews"],
    mode="lines",
    line=dict(color=ACCENT, width=3),
    hovertemplate="Top %{x:.2f}% books<br>Cover %{y:.2f}% reviews<extra></extra>"
))
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color=MUTED, dash="dash"), name="Uniform baseline"))
fig.update_xaxes(title_text="Top x% books by review count")
fig.update_yaxes(title_text="Cumulative % of reviews")
save_fig(fig, "book_long_tail", title="Book popularity long-tail: cumulative review coverage", height=520)
save_table(book_rank, "book_long_tail")

for p in [1, 5, 10, 20]:
    share = book_rank.loc[book_rank["pct_books"] <= p, "pct_reviews"].max()
    print(f"Top {p}% books cover ~{share:.2f}% of reviews")

In [ ]:
# Cell 15 — Rating bias by book review volume

book_volume_bias = run(f"""
WITH b AS (
    SELECT
        book_id,
        COUNT(*) AS n_reviews,
        AVG(CASE WHEN CAST(rating AS INTEGER) BETWEEN 1 AND 5 THEN CAST(rating AS DOUBLE) END) AS avg_rating
    FROM {BASE_SCAN}
    GROUP BY book_id
),
tiered AS (
    SELECT
        *,
        CASE
            WHEN n_reviews BETWEEN 1 AND 10 THEN '1–10'
            WHEN n_reviews BETWEEN 11 AND 50 THEN '11–50'
            WHEN n_reviews BETWEEN 51 AND 200 THEN '51–200'
            WHEN n_reviews BETWEEN 201 AND 1000 THEN '201–1000'
            ELSE '1000+'
        END AS review_volume_tier
    FROM b
)
SELECT
    review_volume_tier,
    COUNT(*) AS books,
    AVG(avg_rating) AS mean_book_rating,
    MEDIAN(avg_rating) AS median_book_rating,
    AVG(n_reviews) AS avg_reviews_per_book
FROM tiered
GROUP BY review_volume_tier
ORDER BY
    CASE review_volume_tier
        WHEN '1–10' THEN 1
        WHEN '11–50' THEN 2
        WHEN '51–200' THEN 3
        WHEN '201–1000' THEN 4
        ELSE 5
    END
""", "rating bias by book volume")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=book_volume_bias["review_volume_tier"], y=book_volume_bias["mean_book_rating"], name="Mean book rating", marker_color=ACCENT, text=book_volume_bias["mean_book_rating"].round(2), textposition="outside"), secondary_y=False)
fig.add_trace(go.Scatter(x=book_volume_bias["review_volume_tier"], y=book_volume_bias["books"], name="# Books", mode="lines+markers", line=dict(color=WARM, width=3)), secondary_y=True)
fig.update_yaxes(title_text="Mean book rating", secondary_y=False)
fig.update_yaxes(title_text="# Books", type="log", secondary_y=True)
fig.update_xaxes(title_text="Book review volume tier")
save_fig(fig, "rating_bias_by_book_volume", title="Rating bias by book review volume", height=500)
save_table(book_volume_bias, "rating_bias_by_book_volume")
display(book_volume_bias)

In [ ]:
# Cell 16 — Review length analysis: char length + word length sample

review_len = run(f"""
SELECT
    CAST(rating AS INTEGER) AS rating,
    review_text,
    LENGTH(review_text) AS char_len,
    {DATE_TS} AS date_ts
FROM {BASE_SCAN}
WHERE review_text IS NOT NULL
  AND LENGTH(TRIM(review_text)) BETWEEN 20 AND 10000
  AND CAST(rating AS INTEGER) BETWEEN 1 AND 5
USING SAMPLE {REVIEW_LENGTH_SAMPLE_N} ROWS
""", f"review length sample {REVIEW_LENGTH_SAMPLE_N:,}")

review_len["word_len"] = review_len["review_text"].astype(str).str.split().str.len()
review_len["year"] = pd.to_datetime(review_len["date_ts"], errors="coerce").dt.year

fig = make_subplots(rows=1, cols=2, subplot_titles=["Character length distribution", "Word length by rating"], column_widths=[0.48, 0.52])
fig.add_trace(go.Histogram(x=review_len["char_len"], nbinsx=100, marker_color=ACCENT, name="char_len"), row=1, col=1)
for r in sorted(review_len["rating"].dropna().unique()):
    sub = review_len.loc[review_len["rating"] == r, "word_len"]
    fig.add_trace(go.Box(y=sub, name=f"{r}★", boxmean=True, showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Characters", row=1, col=1)
fig.update_yaxes(title_text="Reviews", row=1, col=1)
fig.update_yaxes(title_text="Words", row=1, col=2)
save_fig(fig, "review_length", title="Review length distribution and rating relationship", height=520, width=1250)

length_by_rating = review_len.groupby("rating").agg(
    sample_reviews=("rating", "count"),
    median_chars=("char_len", "median"),
    mean_chars=("char_len", "mean"),
    p90_chars=("char_len", lambda x: np.quantile(x, 0.90)),
    median_words=("word_len", "median"),
    mean_words=("word_len", "mean"),
    p90_words=("word_len", lambda x: np.quantile(x, 0.90)),
).reset_index()
save_table(length_by_rating, "review_length_by_rating")
display(length_by_rating)

In [ ]:
# Cell 17 — Review length over time

length_year = (
    review_len
    .dropna(subset=["year"])
    .groupby("year")
    .agg(
        sample_reviews=("rating", "count"),
        median_chars=("char_len", "median"),
        mean_chars=("char_len", "mean"),
        median_words=("word_len", "median"),
        mean_words=("word_len", "mean")
    )
    .reset_index()
    .sort_values("year")
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=length_year["year"], y=length_year["median_words"], mode="lines+markers", name="Median words", line=dict(color=ACCENT, width=3)), secondary_y=False)
fig.add_trace(go.Bar(x=length_year["year"], y=length_year["sample_reviews"], name="Sample reviews", marker_color=MUTED, opacity=0.35), secondary_y=True)
fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text="Median words", secondary_y=False)
fig.update_yaxes(title_text="Sample reviews", secondary_y=True)
save_fig(fig, "review_length_by_year", title="Review length trend over time", height=500)
save_table(length_year, "review_length_by_year")
display(length_year)

In [ ]:
# Cell 18 — Temporal rating drift heatmap

rating_year = run(f"""
SELECT
    YEAR({DATE_TS}) AS yr,
    CAST(rating AS INTEGER) AS rating,
    COUNT(*) AS n
FROM {BASE_SCAN}
WHERE {DATE_TS} IS NOT NULL
  AND CAST(rating AS INTEGER) BETWEEN 1 AND 5
GROUP BY 1, 2
ORDER BY 1, 2
""", "rating by year")

pivot = rating_year.pivot(index="rating", columns="yr", values="n").fillna(0)
pivot_pct = pivot.div(pivot.sum(axis=0), axis=1) * 100

fig = px.imshow(
    pivot_pct,
    color_continuous_scale="Blues",
    labels={"x": "Year", "y": "Rating", "color": "% of reviews"},
    text_auto=".1f",
    aspect="auto",
    title="Temporal rating drift: rating share by year"
)
save_fig(fig, "rating_drift_heatmap", height=450)

yearly_mean_rating = rating_year.groupby("yr").apply(lambda g: (g["rating"] * g["n"]).sum() / g["n"].sum()).reset_index(name="mean_rating")
save_table(rating_year, "rating_by_year")
save_table(yearly_mean_rating, "yearly_mean_rating")
display(yearly_mean_rating)

In [ ]:
# Cell 19 — Positivity ratio over time

positivity = run(f"""
SELECT
    YEAR({DATE_TS}) AS yr,
    COUNT(*) AS reviews,
    SUM(CASE WHEN CAST(rating AS INTEGER) >= 4 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_4plus,
    SUM(CASE WHEN CAST(rating AS INTEGER) = 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_5star,
    AVG(CASE WHEN CAST(rating AS INTEGER) BETWEEN 1 AND 5 THEN CAST(rating AS DOUBLE) END) AS avg_rating
FROM {BASE_SCAN}
WHERE {DATE_TS} IS NOT NULL
  AND CAST(rating AS INTEGER) BETWEEN 1 AND 5
GROUP BY 1
ORDER BY 1
""", "positivity over time")

fig = go.Figure()
fig.add_trace(go.Scatter(x=positivity["yr"], y=positivity["pct_4plus"], name="4–5 star share", mode="lines+markers", line=dict(color=ACCENT, width=3)))
fig.add_trace(go.Scatter(x=positivity["yr"], y=positivity["pct_5star"], name="5-star share", mode="lines+markers", line=dict(color=WARM, width=3)))
fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text="% of reviews")
save_fig(fig, "positivity_over_time", title="Positivity bias over time", height=500)
save_table(positivity, "positivity_over_time")
display(positivity)

In [ ]:
# Cell 20 — Optional: Genre distribution and genre-level rating

if RUN_GENRE_ANALYSIS and "primary_genre" in cols:
    genre_stats = run(f"""
    SELECT
        primary_genre,
        COUNT(*) AS reviews,
        COUNT(DISTINCT user_id) AS users,
        COUNT(DISTINCT book_id) AS books,
        AVG(CASE WHEN CAST(rating AS INTEGER) BETWEEN 1 AND 5 THEN CAST(rating AS DOUBLE) END) AS avg_rating,
        AVG(LENGTH(review_text)) AS avg_char_len
    FROM {BASE_SCAN}
    WHERE primary_genre IS NOT NULL
    GROUP BY 1
    ORDER BY reviews DESC
    """, "genre stats")

    fig = px.bar(
        genre_stats.sort_values("reviews"),
        x="reviews", y="primary_genre",
        orientation="h",
        color="avg_rating",
        color_continuous_scale="Blues",
        hover_data=["users", "books", "avg_char_len"],
        labels={"reviews": "Reviews", "primary_genre": "Genre", "avg_rating": "Avg rating"},
        title="Genre distribution and average rating"
    )
    save_fig(fig, "genre_distribution", height=max(500, 35 * len(genre_stats)))
    save_table(genre_stats, "genre_stats")
    display(genre_stats)
else:
    print("Skip: primary_genre not found.")

In [ ]:
# Cell 21 — Optional: Genre x rating heatmap

if RUN_GENRE_ANALYSIS and "primary_genre" in cols:
    genre_rating = run(f"""
    SELECT
        primary_genre,
        CAST(rating AS INTEGER) AS rating,
        COUNT(*) AS n
    FROM {BASE_SCAN}
    WHERE primary_genre IS NOT NULL
      AND CAST(rating AS INTEGER) BETWEEN 1 AND 5
    GROUP BY 1, 2
    ORDER BY 1, 2
    """, "genre x rating")

    pivot = genre_rating.pivot(index="primary_genre", columns="rating", values="n").fillna(0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    fig = px.imshow(
        pivot_pct,
        color_continuous_scale="Blues",
        labels={"x": "Rating", "y": "Genre", "color": "% within genre"},
        text_auto=".1f",
        aspect="auto",
        title="Rating composition by genre"
    )
    save_fig(fig, "genre_rating_heatmap", height=max(500, 35 * len(pivot_pct)))
    save_table(genre_rating, "genre_rating_distribution")
else:
    print("Skip: primary_genre not found.")

In [ ]:
# Cell 22 — Optional: User genre breadth / bridge users

if RUN_GENRE_ANALYSIS and "primary_genre" in cols:
    user_genre_breadth = run(f"""
    WITH ug AS (
        SELECT user_id, COUNT(DISTINCT primary_genre) AS n_genres
        FROM {BASE_SCAN}
        WHERE primary_genre IS NOT NULL
        GROUP BY user_id
    )
    SELECT
        n_genres,
        COUNT(*) AS users
    FROM ug
    GROUP BY 1
    ORDER BY 1
    """, "user genre breadth")

    user_genre_breadth["pct_users"] = user_genre_breadth["users"] / user_genre_breadth["users"].sum() * 100

    fig = px.bar(
        user_genre_breadth,
        x="n_genres", y="users",
        text=user_genre_breadth["pct_users"].map(lambda x: f"{x:.1f}%"),
        labels={"n_genres": "Distinct genres reviewed by user", "users": "Users"},
        title="User genre breadth"
    )
    fig.update_traces(marker_color=ACCENT, textposition="outside")
    save_fig(fig, "user_genre_breadth", height=500)
    save_table(user_genre_breadth, "user_genre_breadth")
    display(user_genre_breadth)

    if "is_bridge_user" in cols:
        bridge_stats = run(f"""
        SELECT
            CAST(is_bridge_user AS BOOLEAN) AS is_bridge_user,
            COUNT(*) AS reviews,
            COUNT(DISTINCT user_id) AS users,
            AVG(CASE WHEN CAST(rating AS INTEGER) BETWEEN 1 AND 5 THEN CAST(rating AS DOUBLE) END) AS avg_rating,
            AVG(LENGTH(review_text)) AS avg_char_len
        FROM {BASE_SCAN}
        GROUP BY 1
        ORDER BY 1
        """, "bridge user stats")
        save_table(bridge_stats, "bridge_user_stats")
        display(bridge_stats)
else:
    print("Skip: primary_genre not found.")

In [ ]:
# Cell 23 — Optional: Genre overlap network
# Nếu quá nặng, set RUN_GENRE_NETWORK = False ở Cell 1.

if RUN_GENRE_NETWORK and RUN_GENRE_ANALYSIS and "primary_genre" in cols:
    genre_overlap = run(f"""
    WITH ug AS (
        SELECT DISTINCT user_id, primary_genre
        FROM {BASE_SCAN}
        WHERE primary_genre IS NOT NULL
    ),
    pairs AS (
        SELECT
            a.primary_genre AS genre_a,
            b.primary_genre AS genre_b,
            COUNT(*) AS users_overlap
        FROM ug a
        JOIN ug b
          ON a.user_id = b.user_id
         AND a.primary_genre < b.primary_genre
        GROUP BY 1, 2
    )
    SELECT *
    FROM pairs
    ORDER BY users_overlap DESC
    LIMIT 50
    """, "genre overlap pairs")

    genres = sorted(set(genre_overlap["genre_a"]) | set(genre_overlap["genre_b"]))
    idx = {g: i for i, g in enumerate(genres)}

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=18, line=dict(color="rgba(0,0,0,0.2)", width=0.5), label=genres),
        link=dict(
            source=genre_overlap["genre_a"].map(idx),
            target=genre_overlap["genre_b"].map(idx),
            value=genre_overlap["users_overlap"],
            hovertemplate="%{source.label} ↔ %{target.label}<br>Users overlap: %{value:,}<extra></extra>"
        )
    )])
    save_fig(fig, "genre_overlap_network", title="Genre overlap network based on shared users", height=650, width=1200)
    save_table(genre_overlap, "genre_overlap_pairs")
    display(genre_overlap)
else:
    print("Skip: genre network disabled or primary_genre not found.")

In [ ]:
# Cell 24 — Optional: Split diagnostics for preprocessed dataset

if "split" in cols:
    split_stats = run(f"""
    SELECT
        split,
        COUNT(*) AS reviews,
        COUNT(DISTINCT user_id) AS users,
        COUNT(DISTINCT book_id) AS books,
        MIN(CAST({DATE_TS} AS DATE)) AS earliest,
        MAX(CAST({DATE_TS} AS DATE)) AS latest,
        AVG(CASE WHEN CAST(rating AS INTEGER) BETWEEN 1 AND 5 THEN CAST(rating AS DOUBLE) END) AS avg_rating
    FROM {BASE_SCAN}
    GROUP BY split
    ORDER BY
        CASE split
            WHEN 'train' THEN 1
            WHEN 'val' THEN 2
            WHEN 'test' THEN 3
            ELSE 4
        END
    """, "split stats")

    fig = px.bar(
        split_stats,
        x="split", y="reviews",
        text=split_stats["reviews"].map(compact_int),
        hover_data=["users", "books", "earliest", "latest", "avg_rating"],
        labels={"split": "Split", "reviews": "Reviews"},
        title="Preprocessed split distribution"
    )
    fig.update_traces(marker_color=ACCENT, textposition="outside")
    save_fig(fig, "split_distribution", height=480)
    save_table(split_stats, "split_stats")
    display(split_stats)
else:
    print("Skip: split column not found.")

In [ ]:
# Cell 25 — Cold-start diagnostics

cold = run(f"""
WITH u AS (
    SELECT user_id, COUNT(*) AS n_reviews
    FROM {BASE_SCAN}
    GROUP BY user_id
),
b AS (
    SELECT book_id, COUNT(*) AS n_reviews
    FROM {BASE_SCAN}
    GROUP BY book_id
),
summary AS (
    SELECT
        'users' AS entity,
        COUNT(*) AS n_entities,
        SUM(CASE WHEN n_reviews < 5 THEN 1 ELSE 0 END) AS n_lt5,
        SUM(CASE WHEN n_reviews < 10 THEN 1 ELSE 0 END) AS n_lt10,
        AVG(n_reviews) AS avg_reviews,
        MEDIAN(n_reviews) AS median_reviews
    FROM u
    UNION ALL
    SELECT
        'books' AS entity,
        COUNT(*) AS n_entities,
        SUM(CASE WHEN n_reviews < 5 THEN 1 ELSE 0 END) AS n_lt5,
        SUM(CASE WHEN n_reviews < 10 THEN 1 ELSE 0 END) AS n_lt10,
        AVG(n_reviews) AS avg_reviews,
        MEDIAN(n_reviews) AS median_reviews
    FROM b
)
SELECT
    *,
    n_lt5 * 100.0 / n_entities AS pct_lt5,
    n_lt10 * 100.0 / n_entities AS pct_lt10
FROM summary
""", "cold-start diagnostics")

fig = go.Figure()
fig.add_trace(go.Bar(x=cold["entity"], y=cold["pct_lt5"], name="<5 reviews", marker_color=ACCENT, text=cold["pct_lt5"].round(1).astype(str) + "%"))
fig.add_trace(go.Bar(x=cold["entity"], y=cold["pct_lt10"], name="<10 reviews", marker_color=WARM, text=cold["pct_lt10"].round(1).astype(str) + "%"))
fig.update_traces(textposition="outside")
fig.update_yaxes(title_text="% entities")
fig.update_xaxes(title_text="Entity")
fig.update_layout(barmode="group")
save_fig(fig, "cold_start_diagnostics", title="Cold-start diagnostics: low-interaction users/books", height=480)
save_table(cold, "cold_start_diagnostics")
display(cold)

In [ ]:
# Cell 26 — Build web-demo index.html

index_items = []
seen = set()
for title, file in saved_artifacts:
    if file not in seen and file.endswith(".html"):
        index_items.append((title, file))
        seen.add(file)

summary_json = json.dumps(summary_metrics, indent=2, ensure_ascii=False)

card_lines = []
for title, file in index_items:
    label = title.replace("_", " ").title()
    card_lines.append(f'<li><a href="{file}" target="_blank">{label}</a> <code>{file}</code></li>')

html_lines = [
    '<!doctype html>',
    '<html><head><meta charset="utf-8">',
    '<title>Goodreads EDA Dashboard Index</title>',
    '<style>',
    'body{font-family:Inter,system-ui,sans-serif;margin:32px;background:#f8fafc;color:#172033}',
    'a{color:#2563eb;text-decoration:none}',
    'li{margin:10px 0;padding:10px;background:white;border:1px solid #e5e7eb;border-radius:10px}',
    'pre{background:#111827;color:#e5e7eb;padding:16px;border-radius:12px;overflow-x:auto}',
    '</style></head><body>',
    '<h1>Goodreads Interactive EDA</h1>',
    '<p>Generated Plotly HTML charts for web demo embedding.</p>',
    '<h2>Summary metrics</h2>',
    '<pre>' + summary_json + '</pre>',
    '<h2>Charts</h2>',
    '<ul>' + '\n'.join(card_lines) + '</ul>',
    '</body></html>'
]

index_path = OUTPUT_DIR / "index.html"
index_path.write_text('\n'.join(html_lines), encoding="utf-8")
print(f"Dashboard index saved → {index_path}")
display(HTML(f'<a href="{index_path}" target="_blank">Open EDA dashboard index</a>'))

In [ ]:
# Cell 27 — Final checklist

print("EDA finished.")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print("\nGenerated files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -", p.name)

print("""
Recommended charts for web demo:
1. overview.html
2. rating_distribution.html
3. reviews_and_users_by_year.html
4. user_growth.html
5. reviews_per_user.html
6. reviews_per_book.html
7. book_long_tail.html
8. review_length.html
9. rating_drift_heatmap.html
10. genre_distribution.html, if primary_genre exists
""")